# pyrweq 示例：RWEQ 风蚀模拟全流程

本 notebook 演示 pyrweq 的完整工作流（论文复现用途）：
单期计算 → 侵蚀分级 → 防风固沙 → 年度逐月累加 → 分区统计。
数据为随机合成栅格（仅演示流程，非真实研究数据）。


## 0. 准备合成数据

In [ ]:
import os, tempfile
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from pyrweq import compute_rweq, compute_rweq_yearly
from pyrweq.classify import classify_erosion
from pyrweq.sandfix import compute_sandfix
from pyrweq.stats import zonal_stats

rng = np.random.default_rng(42)
tmp = tempfile.mkdtemp()
print('temp dir:', tmp)

def make_tif(name, arr):
    p = os.path.join(tmp, f'{name}.tif')
    with rasterio.open(p, 'w', driver='GTiff', height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype='float32', nodata=-9999.0,
                       tiled=True, blockxsize=64, blockysize=64) as dst:
        dst.write(arr.astype(np.float32), 1)
    return p

shape = (200, 200)
x = np.linspace(0, 1, shape[0])[:, None] * np.linspace(0, 1, shape[1])[None, :]

inputs = dict(
    wind_speed=make_tif('wind',   rng.weibull(2.2, shape) * 6.0),   # 风速 m/s（偏态）
    precip=make_tif('precip',     rng.gamma(2.0, 2.0, shape)),      # 降水 mm
    temp=make_tif('temp',         np.full(shape, 15.0)),            # 气温 °C
    elevation=make_tif('elev',    0.5 + 2.0 * x),                   # 高程 km
    potential_et=make_tif('pet',  rng.gamma(3.0, 2.0, shape)),      # 潜在蒸散 mm
    snow_depth=make_tif('snow',   np.zeros(shape)),                 # 雪深 mm
    sand_content=make_tif('sand', rng.uniform(40, 85, shape)),      # 砂粒 %
    silt_content=make_tif('silt', rng.uniform(10, 35, shape)),      # 粉粒 %
    clay_content=make_tif('clay', rng.uniform(5, 25, shape)),       # 粘粒 %
    organic_matter=make_tif('om', rng.uniform(0.2, 2.0, shape)),    # 有机质 %
    ndvi=make_tif('ndvi',         np.clip(0.15 + 0.5 * x + rng.normal(0, 0.05, shape), 0, 1)),
)
print('11 个输入栅格已生成')

## 1. 单期 RWEQ 计算

In [ ]:
result = compute_rweq(**inputs, output_dir=os.path.join(tmp, 'out'), n_workers=4)
print('SL 均值: %.4f, 最大值: %.4f' % (np.nanmean(result.sl), np.nanmax(result.sl)))
print('因子: WF=%.2f EF=%.3f SCF=%.3f K=%.3f C=%.3f' % (
    np.nanmean(result.wf), np.nanmean(result.ef), np.nanmean(result.scf),
    np.nanmean(result.k_prime), np.nanmean(result.c)))

## 2. 侵蚀强度分级（SL190-2007 六级）

In [ ]:
classes, labels = classify_erosion(result.sl)
import collections
counts = collections.Counter(classes[classes > 0].tolist())
print('分级结果（像元数）:')
for k in sorted(counts):
    print(f'  {k} 级（{labels[k]}）: {counts[k]}')

## 3. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, arr, title in [
    (axes[0], result.sl, '风蚀量 SL (kg/m^2)'),
    (axes[1], result.c, '植被因子 C'),
    (axes[2], classes, '侵蚀强度分级 (SL190-2007)'),
]:
    im = ax.imshow(arr, cmap='YlOrBr' if ax is not axes[2] else 'viridis')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 4. 防风固沙量 G = SL_pot(C=裸土) − SL_actual

In [ ]:
g = compute_sandfix(**inputs, n_workers=4)
print('固沙量 G 均值: %.4f kg/m^2' % np.nanmean(g))
plt.figure(figsize=(6, 4.2))
plt.imshow(g, cmap='Greens')
plt.title('防风固沙量 G (kg/m^2)')
plt.colorbar(fraction=0.046)
plt.show()

## 5. 年度逐月累加

标准 RWEQ 工作流：逐月计算 WF，年风蚀量 = Σ 各月 SL。
这里演示 3 期（每月风速/降水/NDVI 不同，土壤参数不变）。

In [ ]:
def month_inputs(seed, wind_scale, ndvi_shift):
    r = np.random.default_rng(seed)
    m = dict(inputs)  # 复用路径输入（土壤、DEM 等不变）
    m['wind_speed'] = make_tif(f'wind_{seed}', r.weibull(2.2, shape) * 6.0 * wind_scale)
    m['precip'] = make_tif(f'precip_{seed}', r.gamma(2.0, 2.0, shape) * wind_scale)
    m['ndvi'] = make_tif(f'ndvi_{seed}', np.clip(
        0.15 + 0.5 * x + ndvi_shift + r.normal(0, 0.05, shape), 0, 1))
    return m

months = [month_inputs(1, 1.4, 0.05),   # 冬春：风大、植被少
          month_inputs(2, 0.9, 0.0),    # 夏：风中等
          month_inputs(3, 0.6, 0.10)]   # 秋：风小、植被多

yearly = compute_rweq_yearly(months, n_workers=4)
print('年风蚀总量 SL 均值: %.4f kg/m^2' % np.nanmean(yearly.sl))
for i, m in enumerate(yearly.months):
    print(f'  第 {i+1} 期: SL 均值 {np.nanmean(m.sl):.4f}')
print('年均植被因子 C: %.4f' % np.nanmean(yearly.c))

## 6. 分区统计（如按土地利用/行政区）

In [ ]:
zones = make_tif('zones', rng.integers(1, 5, shape))
rows = zonal_stats(yearly.sl, zones, output_csv=os.path.join(tmp, 'zonal.csv'))
print('分区统计（年风蚀量）:')
for r in rows:
    print(f'  区 {r["zone"]}: 像元 {r["count"]}, 均值 {r["mean"]:.4f}, 总量 {r["sum"]:.2f}')

## 7. 性能对比：numpy vs dask

栅格很大、内存紧张时用 ：路径输入懒读、峰值内存大幅降低。
小栅格（本示例）numpy 更快——按需选择。

In [ ]:
import time
t0 = time.perf_counter()
r_np = compute_rweq(**inputs, backend='numpy', n_workers=1)
t_np = time.perf_counter() - t0

t0 = time.perf_counter()
r_da = compute_rweq(**inputs, backend='dask', n_workers=1)
sl_da = r_da.sl.compute()
t_da = time.perf_counter() - t0

print(f'numpy: {t_np:.2f}s   dask(含compute): {t_da:.2f}s')
print('结果一致:', np.allclose(r_np.sl, sl_da, atol=1e-4))